# E08: Proyecto Integrador - Pipeline ETL Modular

Este es el **proyecto integrador del nivel intermedio**. Reúne en una sola aplicación real **todo** lo aprendido en E01-E07:

- **E01**: Comprehensions y generadores.
- **E02**: POO fundamental (encapsulacion, herencia, composicion).
- **E03**: Dunder methods y Protocol (structural typing).
- **E04**: Dataclasses y typing estatico.
- **E05**: Decoradores.
- **E06**: Standard Library (pathlib, logging, csv, json, sqlite3, functools).
- **E07**: Testing con pytest.

Construiremos un **Pipeline ETL** (Extract → Transform → Load) para un repositorio de ventas ficticio, aplicando las buenas practicas de la ingenieria de datos moderna.

## Objetivos del proyecto

1. **Integrar** los conceptos de E01-E07 en una arquitectura cohesiva y profesional.
2. **Modelar** datos de ventas con `@dataclass` y validarlos con `__post_init__`.
3. **Definir contratos** de componentes con `typing.Protocol` (Extractor, Transformador, Cargador).
4. **Escribir codigo tipado** con type hints en todas las funciones y clases.
5. **Instrumentar** el codigo con `logging` profesional y decoradores de timing/retry.
6. **Aplicar** comprehensions, `pathlib`, `csv`, `json`, `functools` y `sqlite3`.
7. **Automatizar** la orquestacion E→T→L con manejo robusto de errores.
8. **Verificar** el resultado con tests `pytest` reutilizables.

> **Criterio de exito**: el mismo pipeline debe poder leer CSV o JSON, limpiar y enriquecer los datos, y volcar el resultado a varios formatos, **sin modificar su estructura interna** cuando cambia la fuente u el destino.

## Requisitos del Pipeline ETL

### E - Extract (Extraccion)

- Leer datos de ventas desde **CSV** y **JSON** ubicados en un directorio de entrada.
- Devolver siempre una lista de objetos de dominio (`RegistroVenta`), con independencia del formato de origen.

### T - Transform (Transformacion)

- **Limpiar**: eliminar registros con valores criticos nulos (por ej. sin monto), normalizar tipos (float/int), eliminar duplicados.
- **Enriquecer**: crear columnas derivadas (por ej. `total = cantidad * precio_unitario`, calculo de impuesto IVA).
- **Agregar**: `groupby` por categoria/cliente y estadisticas (totales, promedio, conteo, min, max).

### L - Load (Carga)

- Escribir el resultado (registros + agregados) en **CSV**, **JSON** y **parquet** (si pandas esta disponible) en un directorio de salida.
- **(Opcional / extension)** Volcar a una base de datos **SQLite** con `sqlite3` de la stdlib.

```
                     PIPELINE ETL
┌────────────┐   ┌──────────────────────┐   ┌──────────────────┐
│   INPUT    │   │     TRANSFORMER      │   │     OUTPUT       │
│  directorio│   │                      │   │  directorio      │
│            │   │  Limpiar (NaN, tipos,│   │                  │
│  ventas.csv│   │      duplicados)     │   │  ventas_norm.csv │
│  ventas.json│─►│  Enriquecer (total,  │──►│  ventas_norm.json│
│            │   │      IVA)            │   │  ventas.parquet  │
│            │   │  Agregar (groupby)   │   │  db: ventas.db   │
└────────────┘   └──────────────────────┘   └──────────────────┘
      E                    T                       L
   Extractor        Transformador              Cargador
   (Protocol)       (Protocol)                 (Protocol)
```

## Diseno y Arquitectura

Principios aplicados:

- **Responsabilidad unica (SRP)**: cada clase hace una sola cosa (`Extractor`, `Transformador`, `Cargador`, `Pipeline`).
- **Duck typing + Protocol**: los componentes se definen por su **contrato** (metodos y firmas), no por herencia. Esto permite intercambiar implementaciones sin tocar el orquestador.
- **Modelo de dominio**: un `@dataclass` inmutable (`frozen=True`) como `RegistroVenta` encapsula la estructura y validacion de los datos.
- **Type hints en todas partes**: `list[RegistroVenta]`, `dict[str, float]`, etc. Documentan el codigo y permiten verificar con `mypy`.
- **Logging profesional**: registro de eventos, advertencias y errores con `logging` en lugar de `print`.
- **Decoradores transversales**: `@timed` para medir duracion y `@retry` para reintentar operaciones volatiles.

## Desarrollo Paso a Paso

Ejecutaremos el proyecto en **9 pasos** ejecutables. Cada celda construye una pieza sobre la anterior, de modo que al final el pipeline completo queda operativo y testeado.

### Paso 0: Dependencias y configuracion

Importamos la stdlib necesaria, configuramos `logging` y creamos un directorio de trabajo temporal donde generaremos los datos de ejemplo.

In [ ]:
import json
import logging
import random
import sqlite3
import time
import csv
from dataclasses import dataclass, field, asdict
from datetime import datetime, date
from functools import wraps
from pathlib import Path
from statistics import mean
from typing import (
    Any, Callable, Iterable, Iterator, Protocol, Sequence,
    TypeVar, runtime_checkable, Final,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
)
log: Final = logging.getLogger("ETL")

print("Configuracion ETL lista.")

In [ ]:
from pathlib import Path

WORK_DIR: Final = Path.cwd() / "_etl_proyecto"
IN_DIR: Final = WORK_DIR / "input"
OUT_DIR: Final = WORK_DIR / "output"
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio de trabajo: {WORK_DIR}")

#### Generar datos de ejemplo

Creamos dos archivos de origen (CSV y JSON) con datos de ventas que incluyen, deliberadamente, valores nulos y duplicados para que el transformador tenga trabajo real.

In [ ]:
import csv
from pathlib import Path

def _generar_ventas(n: int = 120) -> list[dict[str, Any]]:
    random.seed(42)
    clientes: Final = ["Ana", "Luis", "Marta", "Sofia", "Carlos"]
    categorias: Final = ["Electronica", "Libros", "Ropa", "Hogar"]
    ventas: list[dict[str, Any]] = []
    for _ in range(n):
        ventas.append({
            "fecha": date(2025, random.randint(1, 12), random.randint(1, 28)).isoformat(),
            "cliente": random.choice(clientes),
            "categoria": random.choice(categorias),
            "producto": f"Articulo-{random.randint(1, 50)}",
            "cantidad": random.randint(1, 5),
            "precio_unitario": round(random.uniform(10, 500), 2),
        })
    ventas[3]["precio_unitario"] = None     # dato faltante en CSV
    ventas[7]["cantidad"] = None            # dato faltante en JSON
    ventas[10]["cliente"] = None            # registro incompleto (se descartara)
    ventas.append(dict(ventas[0]))          # duplicado exacto
    ventas.append(dict(ventas[20]))         # duplicado exacto
    return ventas

ventas_brutas: Final = _generar_ventas()

with (IN_DIR / "ventas.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["fecha", "cliente", "categoria", "producto", "cantidad", "precio_unitario"])
    writer.writeheader()
    writer.writerows(ventas_brutas)

with (IN_DIR / "ventas.json").open("w", encoding="utf-8") as f:
    json.dump(ventas_brutas, f, ensure_ascii=False, indent=2)

print("Archivos de origen generados:")
for p in IN_DIR.iterdir():
    print(f"  - {p.name} ({p.stat().st_size} bytes)")

### Paso 1: Modelo de datos con dataclasses

Definimos `RegistroVenta` como una `@dataclass` **inmutable y tipada**. Usamos `__post_init__` para **parsear/validar** los tipos y descartar registros malformados.

In [ ]:
from dataclasses import dataclass, field
from datetime import date

@dataclass(frozen=True, slots=True)
class RegistroVenta:
    """Modelo de dominio inmutable para una venta individual."""
    fecha: date
    cliente: str
    categoria: str
    producto: str
    cantidad: int
    precio_unitario: float

    def __post_init__(self) -> None:
        if self.cantidad <= 0:
            raise ValueError(f"Cantidad debe ser positiva: {self.cantidad}")
        if self.precio_unitario < 0:
            raise ValueError(f"Precio no puede ser negativo: {self.precio_unitario}")

    @property
    def total(self) -> float:
        """Columna derivada (enriquecimiento)."""
        return round(self.cantidad * self.precio_unitario, 2)

    @property
    def iva(self) -> float:
        """IVA del 16%% sobre el total."""
        return round(self.total * 0.16, 2)


v: RegistroVenta = RegistroVenta(date(2025, 1, 5), "Ana", "Libros", "Articulo-1", 2, 25.50)
print(v)
print(f"Total: {v.total} | IVA: {v.iva}")

### Paso 2: Contrato del extractor y clases extractoras

Definimos un `Protocol` que establece el **contrato** de cualquier extractor: debe poder `extraer()` una secuencia de `RegistroVenta`. Luego implementamos un extractor capaz de leer **CSV** y **JSON** desde un directorio.

In [ ]:
from typing import Protocol, runtime_checkable
from pathlib import Path

@runtime_checkable
class Extractor(Protocol):
    """Contrato de extraccion: devuelve registros de venta."""
    def extraer(self) -> list[RegistroVenta]:
        ...


def _a_venta(raw: dict[str, Any]) -> RegistroVenta | None:
    """Convierte un dict plano a RegistroVenta; None si falta dato critico."""
    try:
        fecha: date = date.fromisoformat(str(raw["fecha"]))
        cantidad: int = int(raw["cantidad"])
        precio: float = float(raw["precio_unitario"])
        return RegistroVenta(
            fecha=fecha,
            cliente=str(raw["cliente"]).strip(),
            categoria=str(raw["categoria"]).strip(),
            producto=str(raw["producto"]).strip(),
            cantidad=cantidad,
            precio_unitario=precio,
        )
    except (KeyError, TypeError, ValueError):
        return None


class ExtractorFicheros:
    """Extrae registros desde ficheros CSV/JSON dentro de un directorio."""
    def __init__(self, directorio: Path) -> None:
        self.directorio: Path = directorio
        self._leidos: int = 0
        self._descartados: int = 0

    def _leer_csv(self, ruta: Path) -> list[RegistroVenta]:
        with ruta.open("r", encoding="utf-8", newline="") as f:
            filas: Iterable[dict[str, Any]] = csv.DictReader(f)
            return [
                v for v in (self._a_venta_guardada(fila) for fila in filas)
                if v is not None
            ]

    def _leer_json(self, ruta: Path) -> list[RegistroVenta]:
        with ruta.open("r", encoding="utf-8") as f:
            datos: list[dict[str, Any]] = json.load(f)
        return [
            v for v in (self._a_venta_guardada(fila) for fila in datos)
            if v is not None
        ]

    def _a_venta_guardada(self, raw: dict[str, Any]) -> RegistroVenta | None:
        v: RegistroVenta | None = _a_venta(raw)
        if v is None:
            self._descartados += 1
        else:
            self._leidos += 1
        return v

    def extraer(self) -> list[RegistroVenta]:
        registros: list[RegistroVenta] = []
        for ruta in sorted(self.directorio.glob("*")):
            if ruta.suffix == ".csv":
                registros.extend(self._leer_csv(ruta))
                log.info("CSV leido: %s", ruta.name)
            elif ruta.suffix == ".json":
                registros.extend(self._leer_json(ruta))
                log.info("JSON leido: %s", ruta.name)
        log.info("Extraccion: %d registros validos, %d descartados", self._leidos, self._descartados)
        return registros

In [ ]:
extractor: Extractor = ExtractorFicheros(IN_DIR)
datos_brutos: list[RegistroVenta] = extractor.extraer()
print(f"Registros extraidos validos: {len(datos_brutos)}")

### Paso 3: Transformador

El transformador recibe la lista de `RegistroVenta` y la **limpia** (duplicados), la **enriquece** (total/IVA ya son propiedades) y la **agrega** por categoria con comprehensions y `statistics`.

In [ ]:
from typing import Protocol

@runtime_checkable
class Transformador(Protocol):
    def transformar(self, datos: list[RegistroVenta]) -> tuple[list[RegistroVenta], dict[str, Any]]:
        ...


class TransformadorVentas:
    """Limpia, enriquece y agrega las ventas."""
    def transformar(
        self, datos: list[RegistroVenta]
    ) -> tuple[list[RegistroVenta], dict[str, Any]]:
        limpias: list[RegistroVenta] = self._deduplicar(datos)
        agregados: dict[str, Any] = self._agregar(limpias)
        log.info("Transformacion: %d limpias, %d agregadas", len(limpias), len(agregados["categorias"]))
        return limpias, agregados

    @staticmethod
    def _deduplicar(datos: list[RegistroVenta]) -> list[RegistroVenta]:
        """Elimina duplicados preservando el orden (comprehension + set)."""
        vistos: set[RegistroVenta] = set()
        return [v for v in datos if not (v in vistos or vistos.add(v))]

    @staticmethod
    def _agregar(datos: list[RegistroVenta]) -> dict[str, Any]:
        """Agrega estadisticas por categoria y por cliente."""
        por_categoria: dict[str, list[float]] = {}
        for v in datos:
            por_categoria.setdefault(v.categoria, []).append(v.total)

        categorias: dict[str, dict[str, float]] = {
            cat: {
                "ventas": sum(totales),
                "promedio": round(mean(totales), 2),
                "conteo": float(len(totales)),
                "minimo": min(totales),
                "maximo": max(totales),
            }
            for cat, totales in por_categoria.items()
        }

        por_cliente: dict[str, float] = {}
        for v in datos:
            por_cliente[v.cliente] = por_cliente.get(v.cliente, 0.0) + v.total

        return {
            "categorias": categorias,
            "clientes": dict(sorted(por_cliente.items(), key=lambda kv: kv[1], reverse=True)),
            "total_general": round(sum(v.total for v in datos), 2),
            "num_registros": len(datos),
        }


transformador: Transformador = TransformadorVentas()
ventas_limpias, agregados = transformador.transformar(datos_brutos)
print(f"Ventas limpias (sin duplicados): {len(ventas_limpias)}")
print("Agregados:")
import json as _json
print(_json.dumps(agregados, indent=2, ensure_ascii=False))

### Paso 4: Cargador

El cargador escribe los registros limpios y los agregados en **CSV**, **JSON** y **parquet** (si pandas esta disponible). Usamos `pathlib` para garantizar que el directorio de salida exista.

In [ ]:
from typing import Protocol

@runtime_checkable
class Cargador(Protocol):
    def cargar(self, datos: list[RegistroVenta], agregados: dict[str, Any], ruta: Path) -> None:
        ...


class CargadorFicheros:
    """Escribe los resultados en CSV, JSON y (opcionalmente) parquet."""
    def __init__(self, directorio: Path) -> None:
        self.directorio: Path = directorio
        self.directorio.mkdir(parents=True, exist_ok=True)

    def cargar(self, datos: list[RegistroVenta], agregados: dict[str, Any], ruta: Path) -> None:
        self._csv(datos, ruta)
        self._json(agregados, ruta)
        try:
            self._parquet(datos, ruta)
        except ImportError:
            log.warning("pandas no disponible; se omite parquet")

    def _csv(self, datos: list[RegistroVenta], base: Path) -> None:
        destino: Path = self.directorio / f"{base.stem}.csv"
        columnas: list[str] = ["fecha", "cliente", "categoria", "producto", "cantidad", "precio_unitario", "total", "iva"]
        with destino.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=columnas)
            writer.writeheader()
            for v in datos:
                fila: dict[str, Any] = asdict(v)
                fila["fecha"] = v.fecha.isoformat()
                fila["total"] = v.total
                fila["iva"] = v.iva
                writer.writerow(fila)
        log.info("CSV escrito: %s", destino.name)

    def _json(self, agregados: dict[str, Any], base: Path) -> None:
        destino: Path = self.directorio / f"{base.stem}_agregados.json"
        with destino.open("w", encoding="utf-8") as f:
            json.dump(agregados, f, ensure_ascii=False, indent=2)
        log.info("JSON escrito: %s", destino.name)

    def _parquet(self, datos: list[RegistroVenta], base: Path) -> None:
        import pandas as pd  # type: ignore
        df = pd.DataFrame([
            {**asdict(v), "total": v.total, "iva": v.iva, "fecha": v.fecha.isoformat()}
            for v in datos
        ])
        destino: Path = self.directorio / f"{base.stem}.parquet"
        df.to_parquet(destino, index=False)
        log.info("Parquet escrito: %s", destino.name)


cargador: Cargador = CargadorFicheros(OUT_DIR)
cargador.cargar(ventas_limpias, agregados, OUT_DIR / "ventas_norm")
print("Carga completada. Archivos de salida:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  - {p.name} ({p.stat().st_size} bytes)")

### Paso 5: Pipeline orquestador

El `Pipeline` recibe (por inyeccion de dependencias) un `Extractor`, un `Transformador` y un `Cargador`, y orquesta la secuencia completa con `logging` y manejo de errores. Gracias a los `Protocol`, cualquier implementacion compatible es valida.

In [ ]:
class Pipeline:
    """Orquesta E → T → L. Sin conocimiento de los detalles concretos."""
    def __init__(self, extractor: Extractor, transformador: Transformador, cargador: Cargador) -> None:
        self.extractor: Extractor = extractor
        self.transformador: Transformador = transformador
        self.cargador: Cargador = cargador

    def ejecutar(self, salida: Path) -> dict[str, Any]:
        log.info("Pipeline iniciado")
        try:
            datos: list[RegistroVenta] = self.extractor.extraer()
            log.info("ETAPA E completada: %d registros", len(datos))

            limpias, agregados = self.transformador.transformar(datos)
            log.info("ETAPA T completada: %d registros limpios", len(limpias))

            self.cargador.cargar(limpias, agregados, salida)
            log.info("ETAPA L completada")

            resumen: dict[str, Any] = {"registros_extraidos": len(datos), "registros_limpios": len(limpias), **agregados}
            log.info("Pipeline finalizado con exito")
            return resumen

        except Exception as exc:
            log.exception("Fallo en el pipeline: %s", exc)
            raise


print("Clase Pipeline definida.")

### Paso 6: Decoradores `@timed` y `@retry`

Instrumentamos el pipeline con dos decoradores genericos y reutilizables:

- `@timed`: mide y registra la duracion de cualquier funcion.
- `@retry`: reintenta una operacion un numero de veces ante excepciones transitorias.

In [ ]:
from functools import wraps
import time

T = TypeVar("T")

def timed(func: Callable[..., T]) -> Callable[..., T]:
    """Mide y registra el tiempo de ejecucion de una funcion."""
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> T:
        inicio: float = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            duracion: float = time.perf_counter() - inicio
            log.info("%s ejecutada en %.4f s", func.__name__, duracion)
    return wrapper


def retry(intentos: int = 3, retardo: float = 0.1, excepciones: type[Exception] = Exception):
    """Reintenta la funcion `intentos` veces ante `excepciones`."""
    def decorador(func: Callable[..., T]) -> Callable[..., T]:
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> T:
            ultimo: Exception | None = None
            for i in range(1, intentos + 1):
                try:
                    return func(*args, **kwargs)
                except excepciones as exc:
                    ultimo = exc
                    log.warning("Intento %d/%d fallo (%s)", i, intentos, exc)
                    if i < intentos:
                        time.sleep(retardo)
            assert ultimo is not None
            raise ultimo
        return wrapper
    return decorador


@retry(intentos=3)
def operacion_inestable() -> str:
    log.info("Ejecutando operacion inestable...")
    if random.random() < 0.6:
        raise ConnectionError("Fallo transitorio simulado")
    return "OK"


@timed
def operacion_lenta() -> float:
    time.sleep(0.2)
    return sum(i * i for i in range(1000))


print("Decoradores definidos.")
print("Resultado retry:", operacion_inestable())
print("Resultado timed:", operacion_lenta())

### Paso 7: Ejecutar el pipeline completo

Unimos todos los componentes (Extractor → Transformador → Cargador) mediante el `Pipeline`, aplicando `@timed` y `@retry` a la orquestacion completa.

In [ ]:
@timed
@retry(intentos=2)
def ejecutar_pipeline_completo() -> dict[str, Any]:
    pipeline: Pipeline = Pipeline(
        extractor=ExtractorFicheros(IN_DIR),
        transformador=TransformadorVentas(),
        cargador=CargadorFicheros(OUT_DIR),
    )
    return pipeline.ejecutar(OUT_DIR / "resumen_final")


resumen_final: dict[str, Any] = ejecutar_pipeline_completo()
print("\n=== RESUMEN FINAL DEL PIPELINE ===")
print(f"Registros extraidos: {resumen_final['registros_extraidos']}")
print(f"Registros limpios:   {resumen_final['registros_limpios']}")
print(f"Total general:       {resumen_final['total_general']}")
print(f"Categorias:          {list(resumen_final['categorias'].keys())}")

### Paso 8: Testing con pytest

Verificamos los componentes clave con funciones `test_` ejecutables por `pytest`. Cubrimos: modelo de datos, limpieza (duplicados), enriquecimiento y el pipeline completo.

In [ ]:
def test_registro_venta_calcula_total_e_iva() -> None:
    v: RegistroVenta = RegistroVenta(date(2025, 1, 1), "Ana", "Libros", "A", 2, 100.0)
    assert v.total == 200.0
    assert round(v.iva, 2) == 32.0


def test_registro_venta_rechaza_cantidad_invalida() -> None:
    try:
        RegistroVenta(date(2025, 1, 1), "Ana", "Libros", "A", 0, 100.0)
    except ValueError:
        return
    raise AssertionError("Se esperaba ValueError")


def test_dedup_elimina_duplicados() -> None:
    v: RegistroVenta = RegistroVenta(date(2025, 1, 1), "Ana", "Libros", "A", 1, 10.0)
    lista: list[RegistroVenta] = [v, v]
    limpias, _ = TransformadorVentas().transformar(lista)
    assert len(limpias) == 1


def test_extractor_devuelve_tipos_correctos() -> None:
    ex: Extractor = ExtractorFicheros(IN_DIR)
    datos: list[RegistroVenta] = ex.extraer()
    assert all(isinstance(v, RegistroVenta) for v in datos)
    assert len(datos) > 0


print("Tests definidos. Ejecutalos con:  pytest -q E08_proyecto_etl_modular.ipynb")

In [ ]:
def test_pipeline_completo_genera_salidas() -> None:
    import tempfile
    with tempfile.TemporaryDirectory() as tmp:
        out: Path = Path(tmp)
        pipe: Pipeline = Pipeline(
            extractor=ExtractorFicheros(IN_DIR),
            transformador=TransformadorVentas(),
            cargador=CargadorFicheros(out),
        )
        resumen: dict[str, Any] = pipe.ejecutar(out / "ventas_test")
        assert resumen["registros_limpios"] >= 0
        assert (out / "ventas_test.csv").exists()
        assert (out / "ventas_test_agregados.json").exists()


print("Test del pipeline completo definido.")

#### Ejecutar los tests en vivo

Si tienes `pytest` instalado, descomenta y ejecuta la siguiente celda para correr todos los tests del notebook. Tambien puedes invocar la funcion `_ejecutar_tests()` manualmente.

```python
# !pytest -q --tb=short
# o, de forma portable sin pytest
# for nombre, fn in sorted(globals().items()):
#     if nombre.startswith("test_") and callable(fn):
#         fn(); print(f"PASS {nombre}")
```

### Paso 9 (Extension): Cargador a SQLite

Una extension natural del `Cargador` es volcar los registros a una base **SQLite**. Gracias al `Protocol`, basta con crear una nueva clase `CargadorSqlite` que respete el mismo contrato: no hay que tocar el `Pipeline`.

In [ ]:
class CargadorSqlite:
    """Carga los registros en una base SQLite (extension)."""
    def __init__(self, ruta_db: Path) -> None:
        self.ruta_db: Path = ruta_db

    def cargar(self, datos: list[RegistroVenta], agregados: dict[str, Any], ruta: Path) -> None:
        conexion = sqlite3.connect(self.ruta_db)
        try:
            conexion.execute("""
                CREATE TABLE IF NOT EXISTS ventas (
                    fecha TEXT, cliente TEXT, categoria TEXT, producto TEXT,
                    cantidad INTEGER, precio_real REAL, total REAL, iva REAL
                )
            """)
            conexion.executemany(
                "INSERT INTO ventas VALUES (?,?,?,?,?,?,?,?)",
                [
                    (v.fecha.isoformat(), v.cliente, v.categoria, v.producto,
                     v.cantidad, v.precio_unitario, v.total, v.iva)
                    for v in datos
                ],
            )
            conexion.commit()
            filas: int = conexion.execute("SELECT COUNT(*) FROM ventas").fetchone()[0]
            log.info("SQLite: %d filas en %s", filas, self.ruta_db.name)
        finally:
            conexion.close()


cargador_sql: Cargador = CargadorSqlite(OUT_DIR / "ventas.db")
cargador_sql.cargar(ventas_limpias, agregados, OUT_DIR / "sqlite")
print("Base SQLite generada:", OUT_DIR / "ventas.db")

#### Probar la base SQLite

Consultamos la base recien creada para verificar que los datos estan correctamente persistidos.

In [ ]:
conexion = sqlite3.connect(OUT_DIR / "ventas.db")
total_filas = conexion.execute("SELECT COUNT(*) FROM ventas").fetchone()[0]
por_cat: list[tuple[str, float]] = conexion.execute(
    "SELECT categoria, SUM(total) FROM ventas GROUP BY categoria ORDER BY SUM(total) DESC"
).fetchall()
conexion.close()

print(f"Total filas en SQLite: {total_filas}")
print("Ventas por categoria:")
for cat, tot in por_cat:
    print(f"  {cat}: {tot:,.2f}")

#### Extension avanzada: paralelismo (opcional)

Cuando el volumen de datos es grande, la etapa **T** puede paralelizarse procesando por lotes con `concurrent.futures`. La arquitectura por `Protocol` no cambia; solo cambia la implementacion interna del transformador.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def transformar_paralelo(datos: list[RegistroVenta], n_hilos: int = 4) -> list[RegistroVenta]:
    """Dedup paralelo por lotes (ilustrativo de la extension avanzada)."""
    lotes: Iterable[list[RegistroVenta]] = (
        datos[i::n_hilos] for i in range(n_hilos)
    )
    vistos: set[RegistroVenta] = set()
    with ThreadPoolExecutor(max_workers=n_hilos) as pool:
        futuros = [pool.submit(TransformadorVentas._deduplicar, lote) for lote in lotes]
        for futuro in as_completed(futuros):
            vistos.update(futuro.result())
    return list(vistos)


resultado_paralelo: list[RegistroVenta] = transformar_paralelo(datos_brutos)
print(f"Registros unicos tras dedup paralelo: {len(resultado_paralelo)}")

## Posibles extensiones

- **Nuevos formatos**: solo hay que agregar un método mas al `Extractor` o `Cargador` (XML, Parquet, API HTTP) sin tocar el pipeline.
- **SQLite completo**: el `CargadorSqlite` ya esta probado; puede ampliarse con indices, `UPSERT` y particionado por fecha.
- **Paralelismo**: processar las transformaciones en pool de procesos con `concurrent.futures.ProcessPoolExecutor` para grandes volumenes.
- **Incremental**: guardar un state (offset/checksum) para reprocesar solo lo nuevo.
- **Orquestacion cloud**: envolver cada etapa en `@timed`/`@retry` y emitir metricas a un servicio de monitorizacion.
- **Validacion de esquema**: usar `pydantic` o `jsonschema` para validar los diccionarios de entrada contra un esquema declarado.

## Ejercicios extra (2 retos)

**Reto 1 - Cargador a BigQuery/Parquet real (`@retry` + `timed`)**

Crea una nueva clase `CargadorParquetReintento` (o adapta la existente) que: usen `@retry(intentos=4, excepciones=OSError)` para la escritura de parquet y `@timed` para medir cada archivo. Recuerda que un decorador sobre un metodo necesita mantenerse `self`. Sugerencia: aplica los decoradores en una funcion auxiliar interna o en un metodo estatico.

```python
# Pista: los decoradores aplicados a metodos reciben self como primer arg.
# Puedes decorar un @staticmethod interno que no dependa de self.
```

**Reto 2 - Deteccion de anomalias en la etapa T**

Amplia `TransformadorVentas` para que, ademas de agregar por categoria, detecte **ventas atipicas**: aquellas cuyo total supere `promedio + 2 * desviacion` de su categoria. Devuelve una nueva clave `"anomalias"` en el diccionario de agregados.

```python
# Pista: calcula media y desviacion por categoria, luego filtra con comprehension.
# statistics tiene stdev, pero con muestras pequenas usa pstdev o una mediana.
```

## Resumen y leccion aprendida

**Resumen**

- Un **pipeline ETL profesional** se construye con componentes de responsabilidad única y **contratos explícitos** (`Protocol`), no con un script monolitico.
- Las **dataclasses** encapsulan el modelo de datos y su validacion; los **type hints** documentan el contrato y permiten verificar con `mypy`.
- Los **decoradores** (`@timed`, `@retry`) inyectan comportamiento transversal (medicion, resiliencia) sin ensuciar la logica del dominio.
- **`logging`** y el manejo de errores hacen que el pipeline sea observable y robusto en produccion.
- **Comprehensions**, `pathlib`, `csv`, `json`, `sqlite3` y `functools` resuelven la mayoria de las tareas cotidianas de ETL sin dependencias externas.
- **Testing con pytest** sobre componentes aislados garantiza que futuros cambios no rompan el sistema.

**Leccion aprendida**

> La **arquitectura por contratos** es el verdadero superpoder: al definir `Extractor`, `Transformador` y `Cargador` como `Protocol`, el orquestador `Pipeline` nunca se entera de si la fuente es un CSV, una API o una base de datos. **Separar el "que hace" del "como lo hace"** es la diferencia entre un script desechable y un proyecto de ingenieria de datos mantenible, testeable y extensible.